# Semantic Checkpoint Sweep

This notebook runs semantic segmentation testing across many Toy and Graha checkpoints. For each checkpoint, it saves every test sample's model input tensor, hard prediction, target label, and metrics to disk.

The implementation lives in `scripts/python/semantic_seg/semantic_checkpoint_sweep.py` so this notebook and the sbatch workflow use the same logic.

In [ ]:
# Notebook imports
# Generated from standard/third-party imports used throughout this notebook.
import sys
from argparse import Namespace
from datetime import datetime
from pathlib import Path


## Setup

In [1]:

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "semantic_checkpoint_sweep.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
SCRIPTS_PY_DIR = REPO_ROOT / "scripts" / "python"
SCRIPTS_TASK_DIR = SCRIPTS_PY_DIR / "semantic_seg"
for path in [SCRIPTS_TASK_DIR, SCRIPTS_PY_DIR, REPO_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from lfm.full_model.all_tasks.utils.utils import ensure_data_symlink
from semantic_checkpoint_sweep import build_config, discover_checkpoints, run_sweep

## Config

Set `TOY_CHECKPOINT_DIR` and `GRAHA_CHECKPOINT_DIR` after training finishes. Reminder: after rerunning Toy training, confirm the final checkpoint directory structure before launching the full 200-run sweep.

In [2]:
# Data paths
INPUT_ROOT_DIR = None  # Optional source directory for ./data symlink
DATA_ROOT = None  # Leave as None to use notebooks/full_model/data
SIMLINK_DEST = INPUT_ROOT_DIR

# Checkpoint directories. Set these before running a real sweep.
BASE_DIR = Path("/explore/nobackup/people/ajkerr1/Lunar_FM")
BASE_CKPT_DIR = BASE_DIR / "full_model_lfm/lfm/notebooks/full_model/outputs/semantic_seg_comparison/"
TOY_CHECKPOINT_DIR = BASE_CKPT_DIR / "date_2026_07_17-time_15_11_31/checkpoints/toy_model"
GRAHA_CHECKPOINT_DIR = BASE_CKPT_DIR / "date_2026_07_17-time_15_11_31/checkpoints/full_model"
MODELS = ["toy", "graha"]  # Use ["toy"] or ["graha"] for one side only.

# Output root for checkpoint/sample outputs.
date_str = datetime.now().strftime("date_%Y_%m_%d-time_%H_%M_%S")
OUTPUT_ROOT = str(NOTEBOOK_DIR / "outputs" / f"sem_ckpt_sweep/{date_str}")

# Data/model settings should match training.
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6]
TARGET_SIZE = 256
SPATIAL_TRANSFORM = "crop"
BATCH_SIZE = 16
NUM_WORKERS = 10  # Used once while preloading processed test batches.
NORMALIZE_INPUTS = True
MAX_TEST_SAMPLES = None  # Use a small number for smoke tests, e.g. 5.
MAX_CHECKPOINTS = None  # Use a small number for smoke tests, e.g. 1.

# Optional model/pretrain roots.
DINO_CHECKPOINT = None
GRAHA_PRETRAIN_DIR = None
GRAHA_WAC_MODE = "new-wac"  # Use "vis-uv" to reuse pretrained 5-band vis + 2-band uv modalities.
GRAHA_VIS_UV_MERGE_METHOD = "mean"
GRAHA_STATS_BATCH_SIZE = 16
GRAHA_BATCH_SIZE = 16
GRAHA_NUM_WORKERS = 10
PRELOAD_TEST_BATCHES = True
SEED = 42
VERBOSE = False  # Set True to show datamodule/model setup printouts.

## Build Config

In [3]:
ensure_data_symlink(SIMLINK_DEST, NOTEBOOK_DIR / "data")

args = Namespace(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    toy_checkpoint_dir=TOY_CHECKPOINT_DIR,
    graha_checkpoint_dir=GRAHA_CHECKPOINT_DIR,
    models=MODELS,
    band_filter=BAND_FILTER,
    target_size=TARGET_SIZE,
    spatial_transform=SPATIAL_TRANSFORM,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    normalize_inputs=NORMALIZE_INPUTS,
    max_test_samples=MAX_TEST_SAMPLES,
    dino_checkpoint=DINO_CHECKPOINT,
    graha_pretrain_dir=GRAHA_PRETRAIN_DIR,
    graha_wac_mode=GRAHA_WAC_MODE,
    graha_vis_uv_merge_method=GRAHA_VIS_UV_MERGE_METHOD,
    graha_stats_batch_size=GRAHA_STATS_BATCH_SIZE,
    graha_batch_size=GRAHA_BATCH_SIZE,
    graha_num_workers=GRAHA_NUM_WORKERS,
    max_checkpoints=MAX_CHECKPOINTS,
    seed=SEED,
    verbose=VERBOSE,
    preload_test_batches=PRELOAD_TEST_BATCHES,
)

config = build_config(args)
print("Data root:", config.data_root)
print("Output root:", config.output_root)
print("Models:", config.models)
print("Toy checkpoints:", config.toy_checkpoint_dir)
print("Graha checkpoints:", config.graha_checkpoint_dir)

SIMLINK_DEST is None; leaving data symlink unchanged.
Data root: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/data
Output root: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/sem_ckpt_sweep_date_2026_07_20-time_09_28_28
Models: ['toy', 'graha']
Toy checkpoints: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/semantic_seg_comparison/date_2026_07_17-time_15_11_31/checkpoints/toy_model
Graha checkpoints: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/semantic_seg_comparison/date_2026_07_17-time_15_11_31/checkpoints/full_model


## Inspect Checkpoints

Use this cell before running a large sweep to confirm checkpoint discovery and epoch parsing.

In [4]:
if config.toy_checkpoint_dir is not None:
    toy_checkpoints = discover_checkpoints(config.toy_checkpoint_dir, max_checkpoints=config.max_checkpoints)
    print("Toy checkpoints:", len(toy_checkpoints))
    for item in toy_checkpoints[:5]:
        print(item.name, item.epoch, item.path)

if config.graha_checkpoint_dir is not None:
    graha_checkpoints = discover_checkpoints(config.graha_checkpoint_dir, max_checkpoints=config.max_checkpoints)
    print("Graha checkpoints:", len(graha_checkpoints))
    for item in graha_checkpoints[:5]:
        print(item.name, item.epoch, item.path)

Toy checkpoints: 101
epoch_000 0 /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/semantic_seg_comparison/date_2026_07_17-time_15_11_31/checkpoints/toy_model/model-epoch-00-val-loss=0.871.ckpt
epoch_001 1 /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/semantic_seg_comparison/date_2026_07_17-time_15_11_31/checkpoints/toy_model/model-epoch-01-val-loss=0.856.ckpt
epoch_002 2 /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/semantic_seg_comparison/date_2026_07_17-time_15_11_31/checkpoints/toy_model/model-epoch-02-val-loss=0.849.ckpt
epoch_003 3 /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/semantic_seg_comparison/date_2026_07_17-time_15_11_31/checkpoints/toy_model/model-epoch-03-val-loss=0.842.ckpt
epoch_004 4 /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/

## Run Sweep

For smoke tests, set `MAX_CHECKPOINTS = 1` and `MAX_TEST_SAMPLES = 2` in the config cell.

In [5]:
results = run_sweep(config)
results.keys()

[rank: 0] Seed set to 42


[Toy] Found 101 checkpoint(s).


Using cache found in /home/ajkerr1/.cache/torch/hub/facebookresearch_dinov3_main


Toy preload test batches:   0%|          | 0/7 [00:00<?, ?it/s]

[Toy] Preloaded 7 test batch(es).


...

Graha epoch_099 batches:   0%|          | 0/7 [00:00<?, ?it/s]

[Graha] epoch_099: F1=0.5508, IoU=0.3801, samples=100


Graha last batches:   0%|          | 0/7 [00:00<?, ?it/s]

[Graha] last: F1=0.5508, IoU=0.3801, samples=100
Saved model summary to /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/sem_ckpt_sweep_date_2026_07_20-time_09_28_28/graha_model/checkpoint_metrics_summary.txt


dict_keys(['toy', 'graha'])